# TXL-PBC Classical Blood Cell Counter Streamlit App

This notebook is **self-contained**: open it on its own in Colab (or run it locally) and the cells below will clone the project code and the TXL-PBC dataset from GitHub, install dependencies, and launch the Streamlit App through a temporary Cloudflare tunnel. No manual file or zip uploads are required.

The detector uses classical image processing and classical machine learning only. It does not use deep learning.

## 1. Get the project code (git clone)

If the notebook is already running inside the project folder (so `app.py` is next to it), this cell does nothing. Otherwise it shallow-clones the project repo from GitHub and changes into the `Recon-Blood` project directory.

In [ ]:
import os, subprocess

PROJECT_REPO = 'https://github.com/SoWiEee/School-Homeworks.git'
PROJECT_SUBDIR = 'Recon-Blood'  # the project lives in this subfolder of the repo

# Only bootstrap if app.py is not already in the current directory.
if not os.path.exists('app.py'):
    if not os.path.exists(PROJECT_SUBDIR):
        clone_root = 'School-Homeworks'
        if not os.path.exists(clone_root):
            print('Cloning project repo (shallow)...')
            subprocess.check_call(['git', 'clone', '--depth', '1', PROJECT_REPO, clone_root])
        target = os.path.join(clone_root, PROJECT_SUBDIR)
    else:
        target = PROJECT_SUBDIR
    os.chdir(target)

assert os.path.exists('app.py'), f'app.py not found in {os.getcwd()} after setup.'
print('Project directory:', os.getcwd())
print('Files:', sorted(os.listdir('.')))

## 2. Install dependencies

In [ ]:
import os, subprocess, sys

if os.path.exists('requirements.txt'):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'])
else:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'streamlit', 'opencv-python-headless', 'numpy', 'pandas', 'scikit-image', 'scikit-learn', 'pillow', 'matplotlib'])
print('Dependencies installed.')

## 3. Get the TXL-PBC dataset (git clone)

The App can run with the packaged samples, but the grading demo should use the full dataset so that train / val / test selection works properly. This cell shallow-clones the dataset directly from GitHub — no upload needed. If the dataset is already present, it is reused.

In [ ]:
import os, subprocess

DATASET_REPO = 'https://github.com/lugan113/TXL-PBC_Dataset.git'

CANDIDATE_ROOTS = [
    'TXL-PBC_Dataset/TXL-PBC',
    './TXL-PBC_Dataset/TXL-PBC',
    '/content/TXL-PBC_Dataset/TXL-PBC',
]

def find_dataset():
    for root in CANDIDATE_ROOTS:
        if os.path.exists(os.path.join(root, 'images')) and os.path.exists(os.path.join(root, 'labels')):
            return root
    return None

root = find_dataset()
if root is None:
    print('Cloning TXL-PBC dataset (shallow)...')
    subprocess.check_call(['git', 'clone', '--depth', '1', DATASET_REPO, 'TXL-PBC_Dataset'])
    root = find_dataset()

if root:
    print('Dataset root:', root)
else:
    print('Dataset not found after clone; the App will fall back to packaged samples.')

## 4. Quick sanity check

In [ ]:
import os
from blood_cell_detector import list_images, load_image, load_yolo_gt, detect_cells, count_boxes, load_platelet_model

root = None
for candidate in ['TXL-PBC_Dataset/TXL-PBC', '/content/TXL-PBC_Dataset/TXL-PBC', './samples']:
    if os.path.exists(os.path.join(candidate, 'images')) and os.path.exists(os.path.join(candidate, 'labels')):
        root = candidate
        break

paths = list_images(root, 'test') if root else []
print('Root:', root)
print('Number of test images:', len(paths))
if paths:
    img = load_image(paths[0])
    model = load_platelet_model('et_platelet_model.pkl') if os.path.exists('et_platelet_model.pkl') else None
    preds = detect_cells(img, platelet_model=model)
    gts = load_yolo_gt(paths[0], root)
    print('Example image:', os.path.basename(paths[0]))
    print('GT counts:', count_boxes(gts))
    print('Pred counts:', count_boxes(preds))

## 5. Launch Streamlit with cloudflared

Run this cell and open the printed public URL. Keep this notebook running while using the App.

In [ ]:
import subprocess, time, re, os, signal

CF_URL  = 'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64'
CF_BIN  = './cloudflared'

# Stop old app/tunnel processes if this cell is re-run.
for name in ['streamlit_proc', 'cf_proc']:
    proc = globals().get(name)
    if proc is not None and proc.poll() is None:
        proc.terminate()
        time.sleep(1)

# Download cloudflared. If wget fails, fall back to curl.
if not os.path.exists(CF_BIN):
    print('Downloading cloudflared...')
    ok = subprocess.run(['wget', '-q', CF_URL, '-O', CF_BIN]).returncode == 0
    if not ok:
        ok = subprocess.run(['curl', '-sL', CF_URL, '-o', CF_BIN]).returncode == 0
    if not ok:
        raise RuntimeError('cloudflared download failed. Check network settings or use Colab port forwarding instead.')
    subprocess.run(['chmod', '+x', CF_BIN])
    print('cloudflared downloaded.')

# Launch Streamlit in the background.
streamlit_proc = subprocess.Popen(
    ['streamlit', 'run', 'app.py',
     '--server.port=8501',
     '--server.headless=true',
     '--server.enableCORS=false',
     '--server.enableXsrfProtection=false'],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)
time.sleep(3)

# Start Cloudflare tunnel and parse the public URL from stderr.
cf_proc = subprocess.Popen(
    [CF_BIN, 'tunnel', '--url', 'http://localhost:8501'],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.PIPE
)

url = None
for line in cf_proc.stderr:
    text = line.decode('utf-8', errors='ignore')
    match = re.search(r'https://[a-z0-9\-]+\.trycloudflare\.com', text)
    if match:
        url = match.group(0)
        break

print('Streamlit App started.')
print(f'Public URL: {url}')
print('Open the URL above to use the interactive App.')

## 6. Stop the service, if needed

In [ ]:
for name in ['streamlit_proc', 'cf_proc']:
    proc = globals().get(name)
    if proc is not None and proc.poll() is None:
        proc.terminate()
        print('Stopped', name)